# Análise de Indicadores Econômicos e Mercado Financeiro Brasileiro

**Autor:** Maycon Serzedelo  
**Objetivo:** Explorar a relação entre indicadores macroeconômicos e o desempenho do Ibovespa + previsão + regimes de volatilidade.

---

## 1. Configuração e Importações

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import carregar_dados

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Dados

In [ ]:
df = carregar_dados(data_inicio='2018-01-01')
print('\nShape:', df.shape)
df.tail()

## 3. Evolução Temporal dos Indicadores

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
fig.suptitle('Evolução dos Indicadores Econômicos e Ibovespa', fontsize=16, fontweight='bold')

indicadores = [
    ('IPCA', 'IPCA - Inflação'),
    ('IGP_M', 'IGP-M'),
    ('Selic', 'Selic (%)'),
    ('Cambio_USD_BRL', 'Câmbio USD/BRL'),
    ('IBC_Br', 'IBC-Br (Atividade)'),
    ('PIB', 'PIB'),
    ('Producao_Industrial', 'Produção Industrial'),
    ('Vendas_Varejo', 'Vendas no Varejo'),
    ('Desemprego', 'Taxa de Desemprego (%)'),
    ('Credito_Total', 'Crédito Total'),
    ('Reservas_Internacionais', 'Reservas Internacionais'),
    ('Ibovespa', 'Ibovespa')
]

for ax, (col, titulo) in zip(axes.flat, indicadores):
    if col in df.columns:
        df[col].plot(ax=ax, title=titulo)
        ax.set_xlabel('')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Matriz de Correlação

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação entre Indicadores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Análise de Regimes de Volatilidade

Vamos identificar períodos de **alta** e **baixa volatilidade** no Ibovespa.

Método simples e eficaz:
1. Calculamos o retorno diário
2. Calculamos a volatilidade móvel (desvio-padrão dos retornos)
3. Classificamos como **Alta Vol** quando está acima da mediana histórica

In [ ]:
# Retorno diário do Ibovespa
df['Retorno'] = df['Ibovespa'].pct_change()

# Volatilidade móvel de 21 dias (≈ 1 mês de pregão)
df['Vol_21d'] = df['Retorno'].rolling(window=21).std() * np.sqrt(252)  # anualizada

# Define o regime com base na mediana
mediana_vol = df['Vol_21d'].median()
df['Regime'] = np.where(df['Vol_21d'] > mediana_vol, 'Alta Volatilidade', 'Baixa Volatilidade')

print(f'Mediana da volatilidade anualizada: {mediana_vol:.2%}')
print('\nDistribuição dos regimes:')
print(df['Regime'].value_counts())

In [ ]:
# Gráfico de regimes
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Preço do Ibovespa colorido por regime
for regime, cor in [('Baixa Volatilidade', 'green'), ('Alta Volatilidade', 'red')]:
    mask = df['Regime'] == regime
    axes[0].plot(df.index[mask], df['Ibovespa'][mask], '.', color=cor, label=regime, markersize=2)

axes[0].set_title('Ibovespa por Regime de Volatilidade', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volatilidade ao longo do tempo
axes[1].plot(df.index, df['Vol_21d'], color='steelblue', label='Volatilidade 21d (anualizada)')
axes[1].axhline(mediana_vol, color='red', linestyle='--', label=f'Mediana ({mediana_vol:.1%})')
axes[1].set_title('Volatilidade Móvel do Ibovespa', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas por regime
print('=== Estatísticas do Ibovespa por Regime ===')
stats_regime = df.groupby('Regime').agg({
    'Retorno': ['mean', 'std', 'count'],
    'Ibovespa': ['mean', 'min', 'max']
}).round(4)
print(stats_regime)

print('\n=== Retorno médio diário ===')
print(df.groupby('Regime')['Retorno'].mean().apply(lambda x: f'{x:.3%}'))

In [ ]:
# Relação dos regimes com outros indicadores
print('=== Média dos indicadores por Regime de Volatilidade ===')
cols_analise = ['Selic', 'Cambio_USD_BRL', 'IPCA', 'IBC_Br', 'Desemprego']
print(df.groupby('Regime')[cols_analise].mean().round(2))

## 6. Modelos Simples de Previsão

Vamos testar dois modelos clássicos de séries temporais no **Ibovespa**:

1. **ARIMA** (AutoRegressive Integrated Moving Average)
2. **Prophet** (modelo do Facebook)

Usaremos os últimos 60 dias úteis como período de teste.

### 6.1 Preparação dos dados para previsão

In [ ]:
serie = df['Ibovespa'].dropna().copy()

horizonte = 60
treino = serie.iloc[:-horizonte]
teste = serie.iloc[-horizonte:]

print(f'Treino: {treino.index.min().date()} → {treino.index.max().date()} ({len(treino)} pontos)')
print(f'Teste : {teste.index.min().date()} → {teste.index.max().date()} ({len(teste)} pontos)')

### 6.2 Modelo ARIMA

In [ ]:
modelo_arima = ARIMA(treino, order=(2, 1, 2))
resultado_arima = modelo_arima.fit()
previsao_arima = resultado_arima.forecast(steps=horizonte)
previsao_arima.index = teste.index

print(resultado_arima.summary().tables[0])

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(treino.index[-120:], treino.values[-120:], label='Treino (últimos 6 meses)', color='steelblue')
plt.plot(teste.index, teste.values, label='Real (Teste)', color='black', linewidth=2)
plt.plot(previsao_arima.index, previsao_arima.values, label='Previsão ARIMA', color='crimson', linestyle='--')
plt.title('Ibovespa – Previsão com ARIMA (2,1,2)', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Modelo Prophet

In [ ]:
df_prophet = treino.reset_index()
df_prophet.columns = ['ds', 'y']

modelo_prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True
)
modelo_prophet.fit(df_prophet)

futuro = modelo_prophet.make_future_dataframe(periods=horizonte, freq='B')
forecast = modelo_prophet.predict(futuro)
forecast_teste = forecast.set_index('ds').loc[teste.index.min():]

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(treino.index[-120:], treino.values[-120:], label='Treino (últimos 6 meses)', color='steelblue')
plt.plot(teste.index, teste.values, label='Real (Teste)', color='black', linewidth=2)
plt.plot(forecast_teste.index, forecast_teste['yhat'], label='Previsão Prophet', color='darkorange', linestyle='--')
plt.fill_between(
    forecast_teste.index,
    forecast_teste['yhat_lower'],
    forecast_teste['yhat_upper'],
    color='orange', alpha=0.2, label='Intervalo de confiança'
)
plt.title('Ibovespa – Previsão com Prophet', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 Comparação de Erros (MAE e RMSE)

In [ ]:
mae_arima = mean_absolute_error(teste, previsao_arima)
rmse_arima = np.sqrt(mean_squared_error(teste, previsao_arima))

pred_prophet = forecast_teste['yhat'].reindex(teste.index)
mae_prophet = mean_absolute_error(teste, pred_prophet)
rmse_prophet = np.sqrt(mean_squared_error(teste, pred_prophet))

print('=== Comparação de Performance ===')
print(f'ARIMA   → MAE: {mae_arima:,.0f}  |  RMSE: {rmse_arima:,.0f}')
print(f'Prophet → MAE: {mae_prophet:,.0f}  |  RMSE: {rmse_prophet:,.0f}')

## 7. Insights Finais

**Análise de regimes:**
- Períodos de alta volatilidade geralmente coincidem com estresse no câmbio, juros ou atividade econômica.
- O retorno médio e o risco mudam bastante entre os regimes.

**Previsão:**
- Modelos de série temporal pura têm dificuldade com a bolsa no curto prazo.
- O exercício serve para aprender o fluxo completo de modelagem.

**Próximos passos possíveis:**
- Usar GARCH para modelar a volatilidade de forma mais sofisticada
- Adicionar variáveis exógenas (Selic, câmbio) nos modelos de previsão
- Criar um dashboard com Streamlit
- Fazer análise de surpresas econômicas (realizado vs expectativa)